In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 한글 그래프 폰트 설정 도우미는 선택 사항입니다. 설치되어 있으면 사용합니다.
try:
    import koreanize_matplotlib
except ImportError:
    print("koreanize_matplotlib이 없어 기본 글꼴로 진행합니다. 한글이 깨지면 pip install koreanize-matplotlib 후 커널을 다시 시작하세요.")

import geopandas as gpd

plt.style.use('default')

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 서울 열린데이터광장 Open API를 통해 원자료 수집
# 분석의 재현성과 실행시간을 고려하여 수집한 데이터를 CSV로 저장 후 사용
from pathlib import Path

DATA_DIR = Path(".")
if not (DATA_DIR / "seoul_real_estate_2024_202606.csv").exists():
    if (DATA_DIR / "HW2" / "seoul_real_estate_2024_202606.csv").exists():
        DATA_DIR = DATA_DIR / "HW2"
    else:
        raise FileNotFoundError("HW2 데이터 CSV 파일을 찾을 수 없습니다.")

seoul = pd.read_csv(DATA_DIR / "seoul_real_estate_2024_202606.csv")

seoul.head()

,RCPT_YR,CGG_CD,CGG_NM,STDG_CD,STDG_NM,LOTNO_SE,LOTNO_SE_NM,MNO,SNO,BLDG_NM,...,THING_AMT,ARCH_AREA,LAND_AREA,FLR,RGHT_SE,RTRCN_DAY,ARCH_YR,BLDG_USG,DCLR_SE,OPBIZ_RESTAGNT_SGG_NM
0,2024,11500,강서구,10300,화곡동,1.0,대지,355.0,20.0,베아트리스,...,23000,29.97,18.0,2.0,NaN,NaN,2017.0,연립다세대,중개거래,서울 강서구
1,2024,11260,중랑구,10100,면목동,NaN,NaN,NaN,NaN,NaN,...,65000,82.85,92.0,NaN,NaN,NaN,1978.0,단독다가구,중개거래,서울 중랑구
2,2024,11350,노원구,10500,상계동,1.0,대지,1285.0,0.0,동아불암,...,70000,114.24,0.0,4.0,NaN,NaN,1999.0,아파트,중개거래,서울 노원구
3,2024,11620,관악구,10100,봉천동,1.0,대지,1712.0,0.0,관악드림(삼성),...,82000,60.00,0.0,19.0,NaN,NaN,2003.0,아파트,중개거래,서울 관악구
4,2024,11230,동대문구,10400,전농동,1.0,대지,103.0,50.0,(103-50),...,24000,44.04,20.0,-1.0,NaN,NaN,1991.0,연립다세대,중개거래,서울 동대문구


In [26]:
dong = gpd.read_file(
    DATA_DIR / "admstr_zone_lgldong_bndry_24.shp",
    encoding="cp949"
)

dong[["EMD_CD", "EMD_NM"]].head()

,EMD_CD,EMD_NM
0,11110101,청운동
1,11110102,신교동
2,11110103,궁정동
3,11110104,효자동
4,11110105,창성동


In [27]:
seoul["CGG_CD"] = seoul["CGG_CD"].astype(str).str.zfill(5)
seoul["STDG_CD"] = seoul["STDG_CD"].astype(str).str.zfill(5)

seoul["EMD_CD"] = (
    seoul["CGG_CD"] +
    seoul["STDG_CD"].str[:3]
)

## STEP 1. 분석용 거래 데이터 정제

원본 `seoul`은 보존하고 `seoul_analysis`를 별도로 만듭니다. 취소 거래를 제외하고 날짜·금액·면적을 정리한 뒤 거래별 만원/평을 계산합니다.

> **코드 확인 사항:** 안내된 `STDG_CD.str[-3:]`는 실제 CSV와 경계 코드에 대조했을 때 0% 매칭을 만들었습니다. `STDG_CD` 값(예: `10300`)의 앞 세 자리로 경계 코드(`11500103`)와 일치하는 것을 확인해 아래에서는 `.str[:3]`을 사용합니다.


In [ ]:
# 원본 seoul은 보존하고 분석용 복사본에서만 정제합니다.
seoul_analysis = seoul.copy()

# CSV에서 날짜가 문자열로 읽힐 수 있으므로 datetime으로 명시 변환합니다.
seoul_analysis["CTRT_DAY"] = pd.to_datetime(
    seoul_analysis["CTRT_DAY"], errors="coerce"
)

# 취소일이 입력된 거래는 분석에서 제외합니다. 공백 문자열도 결측으로 처리합니다.
cancelled = (
    seoul_analysis["RTRCN_DAY"].notna()
    & seoul_analysis["RTRCN_DAY"].astype(str).str.strip().ne("")
)
seoul_analysis = seoul_analysis.loc[~cancelled].copy()

# 금액/면적이 숫자형이 아니거나 변환할 수 없는 값이면 NaN으로 둡니다.
for col in ["THING_AMT", "ARCH_AREA"]:
    seoul_analysis[col] = pd.to_numeric(seoul_analysis[col], errors="coerce")

# 분석용 코드를 문자열 8자리로 정규화합니다.
seoul_analysis["CGG_CD"] = seoul_analysis["CGG_CD"].astype(str).str.zfill(5)
seoul_analysis["STDG_CD"] = seoul_analysis["STDG_CD"].astype(str).str.zfill(5)
seoul_analysis["EMD_CD"] = (
    seoul_analysis["CGG_CD"] + seoul_analysis["STDG_CD"].str[:3]
)

# 1평 = 3.3058㎡. 면적이 양수이고 금액이 존재하는 거래만 계산합니다.
valid_area = seoul_analysis["ARCH_AREA"].gt(0)
valid_amount = seoul_analysis["THING_AMT"].notna()
seoul_analysis["price_per_pyeong"] = np.nan
seoul_analysis.loc[valid_area & valid_amount, "price_per_pyeong"] = (
    seoul_analysis.loc[valid_area & valid_amount, "THING_AMT"]
    / seoul_analysis.loc[valid_area & valid_amount, "ARCH_AREA"]
    * 3.3058
)

# 기본 품질 점검: 제외/결측/비정상 값의 규모를 확인합니다.
print(f"원본 행 수: {len(seoul):,}")
print(f"취소 거래 제외 수: {int(cancelled.sum()):,}")
print(f"분석용 행 수: {len(seoul_analysis):,}")
print("주요 결측치 수:")
print(seoul_analysis[["CTRT_DAY", "THING_AMT", "ARCH_AREA", "price_per_pyeong"]].isna().sum())
print("0 이하 면적 거래 수:", int(seoul_analysis["ARCH_AREA"].le(0).sum()))
print("0 이하 거래금액 수:", int(seoul_analysis["THING_AMT"].le(0).sum()))
print("평당가격 기술통계:")
print(seoul_analysis["price_per_pyeong"].describe())


**확인할 출력:** 취소 거래 제외 수와 분석용 행 수, 날짜/금액/면적 결측 수, 0 이하 면적·금액 수, 평당가격 기술통계를 확인합니다. 거래금액 이상치나 변환 실패가 있으면 아래 집계 전에 원자료를 점검하세요.


## STEP 2. `EMD_CD` 매칭 검증

코드로만 경계를 연결합니다. 전체 분석 거래 기준의 매칭률을 출력하고, 매칭되지 않은 코드는 자치구명·법정동명과 함께 별도 표로 확인합니다.


In [ ]:
# 경계 코드도 문자열 8자리로 통일합니다.
dong["EMD_CD"] = dong["EMD_CD"].astype(str).str.strip().str.zfill(8)

# 거래 코드 중 경계 데이터에 존재하는 코드의 거래 건수를 계산합니다.
dong_codes = set(dong["EMD_CD"].dropna().unique())
matched = seoul_analysis["EMD_CD"].isin(dong_codes)
match_count = int(matched.sum())
total_count = len(seoul_analysis)
match_rate = match_count / total_count if total_count else np.nan
print(f"매칭 거래: {match_count:,} / {total_count:,} ({match_rate:.2%})")

# 미매칭 코드와 거래의 자치구/법정동 이름을 확인하기 위한 목록입니다.
unmatched_emd = (
    seoul_analysis.loc[~matched]
    .groupby(["EMD_CD", "CGG_NM", "STDG_NM"], dropna=False)
    .size()
    .reset_index(name="transaction_count")
    .sort_values("transaction_count", ascending=False)
)
print(f"미매칭 코드 수: {unmatched_emd['EMD_CD'].nunique():,}")
display(unmatched_emd)


**확인할 출력:** 매칭률이 충분히 높은지 확인합니다. `unmatched_emd`가 비어 있지 않으면 코드 체계·선행 0·법정동 경계의 기준연도를 먼저 확인하세요. 미매칭 행은 지도에는 표시되지 않지만 거래 집계에는 남습니다.


## STEP 3. 법정동별 거래 집계

거래량은 취소되지 않은 모든 분석 거래를 셉니다. 평당가격 대표값은 면적이 양수이고 금액이 계산된 거래만 사용합니다. `CTRT_DAY`를 보존한 원본 분석 테이블도 남겨두어 이후 연/월 필터를 적용할 수 있습니다.


In [ ]:
# 거래량 집계: 평당가격 계산이 불가능한 거래도 유효한 거래 건수로 셉니다.
transaction_counts = (
    seoul_analysis
    .groupby(["EMD_CD", "CGG_NM", "STDG_NM"], dropna=False)
    .size()
    .rename("transaction_count")
)

# 평당가격 집계는 계산 가능한 거래만 사용합니다.
price_summary = (
    seoul_analysis.dropna(subset=["price_per_pyeong"])
    .groupby(["EMD_CD", "CGG_NM", "STDG_NM"], dropna=False)["price_per_pyeong"]
    .agg(
        median_price_per_pyeong="median",
        mean_price_per_pyeong="mean"
    )
)

dong_summary = (
    transaction_counts.to_frame()
    .join(price_summary, how="left")
    .reset_index()
)

print("법정동 집계 행 수:", len(dong_summary))
print("평당가격 집계가 없는 법정동 수:", int(dong_summary["median_price_per_pyeong"].isna().sum()))
display(dong_summary.head())


**확인할 출력:** `dong_summary`에 요청한 6개 컬럼이 있는지 확인합니다. `median_price_per_pyeong` 결측은 해당 법정동에 유효한 면적·금액 조합이 없다는 뜻입니다.


## STEP 4. 집계 결과와 법정동 경계 결합

`EMD_CD`를 기준으로 결합합니다. 경계 기준 행을 유지하는 left merge를 사용하므로 거래가 없는 법정동도 지도에 남고 지표는 결측으로 표시됩니다.


In [ ]:
# geometry 외에 집계에 필요한 법정동 이름도 함께 남깁니다.
dong_map = dong.merge(
    dong_summary,
    on="EMD_CD",
    how="left",
    validate="many_to_one" if dong["EMD_CD"].duplicated().any() else "one_to_one"
)

print("경계 행 수:", len(dong))
print("결합 후 지도 행 수:", len(dong_map))
print("거래가 매칭된 경계 수:", int(dong_map["transaction_count"].notna().sum()))
print("좌표계:", dong_map.crs)


**확인할 출력:** 결합 후 지도 행 수와 CRS를 확인합니다. 동일 `EMD_CD`가 경계에 여러 행으로 있는 경우에는 한 코드가 여러 폴리곤으로 나뉘어 표시될 수 있습니다.


## STEP 5. 거래량·평당가격 5분위 단계구분도

`q=5` 분위 구간을 사용합니다. 값이 반복되어 분위 경계가 겹치면 `duplicates="drop"`으로 가능한 구간 수만큼 자동 조정하고, 실제 구간 수를 출력합니다.


In [ ]:
# qcut은 동일한 값이 많아 경계가 중복되면 구간 수가 5보다 적을 수 있습니다.
def make_quantile_bins(series, q=5):
    """수치열을 분위 구간으로 나누고 실제 생성된 구간 수를 반환합니다."""
    valid = series.dropna()
    result = pd.Series(index=series.index, dtype="object")
    if valid.empty:
        return result, 0
    try:
        bins = pd.qcut(valid, q=q, duplicates="drop")
        result.loc[valid.index] = bins.astype(str)
        return result, len(bins.cat.categories)
    except ValueError as exc:
        print("분위 구간 생성 실패:", exc)
        return result, 0

# 두 지표별 구간을 각각 생성합니다.
dong_map["transaction_quantile"], transaction_bins = make_quantile_bins(
    dong_map["transaction_count"]
)
dong_map["price_quantile"], price_bins = make_quantile_bins(
    dong_map["median_price_per_pyeong"]
)
print(f"거래건수 분위 구간 수: {transaction_bins}")
print(f"평당가격 분위 구간 수: {price_bins}")

# 두 지도를 나란히 그립니다. 값이 없는 경계는 연한 회색으로 표시합니다.
fig, axes = plt.subplots(1, 2, figsize=(18, 10))
dong_map.plot(
    column="transaction_quantile", categorical=True, legend=True,
    cmap="YlOrRd", missing_kwds={"color": "lightgrey", "label": "자료 없음"},
    edgecolor="white", linewidth=0.15, ax=axes[0]
)
axes[0].set_title("법정동별 거래건수 (5분위)")
axes[0].set_axis_off()

dong_map.plot(
    column="price_quantile", categorical=True, legend=True,
    cmap="YlGnBu", missing_kwds={"color": "lightgrey", "label": "자료 없음"},
    edgecolor="white", linewidth=0.15, ax=axes[1]
)
axes[1].set_title("법정동별 평당가격 중앙값 (만원/평, 5분위)")
axes[1].set_axis_off()

plt.tight_layout()
plt.show()


## STEP 6. 전세 데이터 수집 및 CSV 저장

서울시 전월세 Open API를 `RCPT_YR`별로 조회합니다. API 키는 프로젝트 루트 `.env`에서 읽고 셀 출력이나 저장 파일에 표시하지 않습니다. `CTRT_DAY` 계약일이 2025-06-01~2026-06-30이고 `RENT_SE`가 `전세`인 행만 남깁니다.

현재 API는 전체 데이터가 약 264만 건이고 요청당 최대 1,000건입니다. 일일 호출 한도를 넘기지 않도록 하루 최대 900페이지까지만 받고, 여러 번 실행해도 그날 수집량을 추적해 다음 날 이어받게 했습니다. `requests`와 `python-dotenv`가 없다면 `pip install requests python-dotenv`로 설치하세요.


In [5]:
from pathlib import Path
import json
import os
import re
import time
import requests
import pandas as pd
from dotenv import load_dotenv

# 노트북 실행 위치가 프로젝트 루트 또는 HW2 폴더인 경우를 모두 지원합니다.
WORK_DIR = Path.cwd()
if WORK_DIR.name.lower() == "hw2":
    DATA_DIR = WORK_DIR
elif (WORK_DIR / "HW2").is_dir():
    DATA_DIR = WORK_DIR / "HW2"
else:
    DATA_DIR = WORK_DIR

env_candidates = [WORK_DIR / ".env", WORK_DIR.parent / ".env", DATA_DIR / ".env"]
env_path = next((path for path in env_candidates if path.is_file()), None)
if env_path is None:
    raise FileNotFoundError("프로젝트 루트의 .env 파일을 찾을 수 없습니다.")
load_dotenv(dotenv_path=env_path, override=False)
SEOUL_API_KEY = os.environ.get("SEOUL_API_KEY", "").strip()
if not SEOUL_API_KEY:
    raise ValueError(".env에 SEOUL_API_KEY가 설정되어 있지 않습니다.")

SERVICE_NAME = "tbLnOpendataRentV"
API_BASE = "http://openapi.seoul.go.kr:8088"
TARGET_YEARS = [2024, 2025, 2026]  # 계약일과 접수연도가 다를 수 있어 모두 확인합니다.
PAGE_SIZE = 1000
MAX_PAGES_PER_DAY = 900
START_DAY, END_DAY = "20250601", "20260630"
JEONSE_CSV_PATH = DATA_DIR / "seoul_jeonse_20250601_20260630.csv"
PARTIAL_CSV_PATH = DATA_DIR / "seoul_jeonse_20250601_20260630.partial.csv"
STATE_PATH = DATA_DIR / "seoul_jeonse_20250601_20260630_progress.json"

# 진행 상태가 있으면 이어받고, 없으면 새 수집을 시작합니다.
if STATE_PATH.exists():
    state = json.loads(STATE_PATH.read_text(encoding="utf-8"))
else:
    state = {"year_index": 0, "next_start": 1, "written_rows": 0}
    PARTIAL_CSV_PATH.unlink(missing_ok=True)
    JEONSE_CSV_PATH.unlink(missing_ok=True)


def fetch_rent_page(year, start_index, end_index):
    """서울시 API의 RCPT_YR 경로 필터를 적용해 한 페이지를 가져옵니다."""
    url = (f"{API_BASE}/{SEOUL_API_KEY}/json/{SERVICE_NAME}/"
           f"{start_index}/{end_index}/{year}")
    for attempt in range(1, 4):
        try:
            response = requests.get(url, timeout=60)
            response.raise_for_status()
            payload = response.json()
        except requests.RequestException as exc:
            if attempt == 3:
                raise RuntimeError(f"{year}년 API 요청 실패 ({type(exc).__name__}).") from None
            time.sleep(attempt * 2)
            continue
        except ValueError:
            raise RuntimeError(f"{year}년 API 응답을 JSON으로 읽지 못했습니다.") from None
        service = payload.get(SERVICE_NAME)
        if service is None:
            result = payload.get("RESULT", {})
            raise RuntimeError(f"{year}년 API 오류 ({result.get('CODE', '오류')}): "
                               f"{result.get('MESSAGE', '응답을 확인하세요.')}")
        return service


def save_page(rows):
    """기간 내 전세 행만 임시 CSV에 추가합니다."""
    if not rows:
        return 0
    frame = pd.DataFrame(rows)
    frame = frame[
        (frame["RENT_SE"].astype(str).str.strip() == "전세")
        & (frame["CTRT_DAY"].astype(str).str.replace(r"\D", "", regex=True).str[:8].between(START_DAY, END_DAY))
    ]
    if frame.empty:
        return 0
    frame.to_csv(PARTIAL_CSV_PATH, mode="a", index=False,
                 header=not PARTIAL_CSV_PATH.exists(),
                 encoding="utf-8-sig" if not PARTIAL_CSV_PATH.exists() else "utf-8")
    return len(frame)


# 여러 번 실행해도 하루 총 요청 수가 900페이지를 넘지 않게 관리합니다.
today = str(pd.Timestamp.now().date())
if state.get("run_day") != today:
    state["run_day"] = today
    state["pages_today"] = 0
pages_this_run = 0
while (state["year_index"] < len(TARGET_YEARS)
       and pages_this_run < MAX_PAGES_PER_DAY
       and state.get("pages_today", 0) < MAX_PAGES_PER_DAY):
    year = TARGET_YEARS[state["year_index"]]
    # 첫 호출에서 해당 접수연도의 전체 건수와 CSV 컬럼을 확인합니다.
    if "total_count" not in state or state.get("active_year") != year:
        first = fetch_rent_page(year, 1, PAGE_SIZE)
        state["total_count"] = int(first.get("list_total_count", 0))
        state["pages_today"] = state.get("pages_today", 0) + 1
        state["active_year"] = year
        state["next_start"] = 1
        print(f"{year} 접수연도 전체 {state['total_count']:,}건")
    start = int(state["next_start"])
    if start > state["total_count"]:
        state["year_index"] += 1
        state.pop("total_count", None)
        state.pop("active_year", None)
        state["next_start"] = 1
        continue
    end = min(start + PAGE_SIZE - 1, state["total_count"])
    page = fetch_rent_page(year, start, end)
    state["written_rows"] = int(state.get("written_rows", 0)) + save_page(page.get("row", []))
    state["pages_today"] = state.get("pages_today", 0) + 1
    state["next_start"] = end + 1
    pages_this_run += 1
    # 진행 상황은 매 페이지 기록해 중단되어도 다음 실행에서 이어받습니다.
    STATE_PATH.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")
    if pages_this_run % 50 == 0:
        print(f"이번 실행 {pages_this_run:,}페이지 · 대상 전세 {state['written_rows']:,}건")

if state["year_index"] >= len(TARGET_YEARS):
    if PARTIAL_CSV_PATH.exists():
        PARTIAL_CSV_PATH.replace(JEONSE_CSV_PATH)
        jeonse = pd.read_csv(JEONSE_CSV_PATH, low_memory=False)
        print(f"수집 완료 · 기간 {START_DAY[:4]}-{START_DAY[4:6]}-{START_DAY[6:]} ~ {END_DAY[:4]}-{END_DAY[4:6]}-{END_DAY[6:]}")
        print(f"전세 계약 {len(jeonse):,}건 · 저장 위치: {JEONSE_CSV_PATH.resolve()}")
        display(jeonse.head())
    else:
        raise ValueError("기간 내 전세 데이터가 없습니다. RENT_SE와 CTRT_DAY 응답을 확인하세요.")
    STATE_PATH.unlink(missing_ok=True)
else:
    print(f"오늘 수집 {state.get('pages_today', 0):,}/ {MAX_PAGES_PER_DAY:,}페이지 · 누적 이번 실행 {pages_this_run:,}페이지")
    print("진행 상태가 저장되었습니다. 다음 날 이 셀을 다시 실행하면 이어받습니다.")
    print(f"현재 임시 파일: {PARTIAL_CSV_PATH.name} (완료 전에는 HTML에서 사용하지 마세요.)")

# 인증키를 노트북 커널 환경에서 제거합니다.

del SEOUL_API_KEY
os.environ.pop("SEOUL_API_KEY", None)


2024 접수연도 전체 1,114,387건
이번 실행 50페이지 · 대상 전세 25건
이번 실행 100페이지 · 대상 전세 25건
이번 실행 150페이지 · 대상 전세 25건


KeyboardInterrupt: 

**실행 확인:** 하루 최대 900페이지를 받고 진행 상태를 저장합니다. 수집이 끝나지 않으면 다음 날 이 셀을 다시 실행해 이어받으세요. 완료 메시지 뒤 `seoul_jeonse_20250601_20260630.csv`를 `jeonse.html`에서 선택하면 됩니다. 임시 CSV는 전체 수집 전까지 대시보드에 사용하지 않습니다.
